In [5]:
import torch
from molhiv.utils import calculate_label_imbalance, prec, acc, rec, roc_auc, download_graph_prop_pred_dataset
from molhiv.model import GATNN
from molhiv.training import Metric, train_val
import torch.nn as nn

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
params = {
    "batch_size": 64,
    "datasize": 100,
    "in_channel": 32,
    "hidden_channel": 256,
    "out_channel": 2,
    "dropout": 0.2,
    "lr": 0.001,
    "weight_decay": 0.001,
    "epochs": 50,
    "datasetname": "ogbg-molhiv",
    "Conv": "GATConv",
    "n_heads": 4,
    "add_self_loops": True,
}

device = "cuda" if torch.cuda.is_available() else "mps"
print(device)
dataset = download_graph_prop_pred_dataset()

split_idx = dataset.get_idx_split()

from torch_geometric.loader import DataLoader
size = params["datasize"]

train_dataset = dataset[split_idx["train"]][:size]
val_dataset = dataset[split_idx["valid"]][:size]

sample_labels = [dataset[i].y.squeeze() for i in range(len(train_dataset))]
class_weights, sample_weights = calculate_label_imbalance(sample_labels)

from torch.utils.data import WeightedRandomSampler
sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(train_dataset, batch_size=params["batch_size"], sampler=sampler)
val_loader = DataLoader(val_dataset, batch_size=params["batch_size"])

model = GATNN(
    params["in_channel"],
    params["hidden_channel"],
    params["out_channel"],
    params["n_heads"],
    params["dropout"],
    params["add_self_loops"]
)
optimizer = torch.optim.Adam(model.parameters(), lr=params["lr"], weight_decay=params["weight_decay"])

metrics = [
    Metric("train_prec", fn=prec, split="train"),
    Metric("val_prec", fn=prec, split="val"),

    Metric("train_acc", fn=acc, split="train"),
    Metric("val_acc", fn=acc, split="val"),

    Metric("train_rec", fn=rec, split="train"),
    Metric("val_rec", fn=rec, split="val"),

    Metric("train_roc_auc", fn=roc_auc, split="train"),
    Metric("val_roc_auc", fn=roc_auc, split="val")
]

criterion = nn.CrossEntropyLoss(weight=class_weights)

# import mlflow
# mlflow.set_tracking_uri("http://127.0.0.1:5000")
# mlflow.set_experiment("Molhiv-GCN-HIV-binding")
# with mlflow.start_run(run_name="testing-mps"):

#     mlflow.log_params(params)

#     for epoch in range(params["epochs"]):
#         results = train_val(model, train_loader, val_loader, optimizer, criterion, metrics)
#         mlflow.log_metrics(results, step=epoch)

mps
